# ROS 2 Subscribers and Diagnosis

This notebook uses local publisher and subscriber starters to diagnose a missing message in a fixed order.

## Start with three prepared terminals

In the first terminal, build the current source, source the overlay, and start the publisher:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
cd "$COLCON_WS"
colcon build --packages-select ros2_pubsub
source install/setup.bash
ros2 run ros2_pubsub my_publisher
```

Keep the publisher running. In a second terminal, source the same ROS 2 distribution and workspace overlay:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
cd "$COLCON_WS"
source install/setup.bash
```

## Read and run the subscriber

Open `packages/ros2_pubsub/ros2_pubsub/my_subscriber.py`. The starter uses `MSG_TYPE = String` to match the publisher. Leave `TOPIC_TO_SUBSCRIBE` and `NODE_NAME` at their initial values for the first run: `chatter` and `listener`. Its subscription passes the callback itself to `create_subscription`; ROS 2 calls that callback for each received message.

`String` stores its text in the `data` field. The callback logs `I heard:` followed by that text. `rclpy.spin()` keeps the subscriber alive long enough to receive messages and call its callback. Start the subscriber in the second terminal:

```bash
ros2 run ros2_pubsub my_subscriber
```

When both nodes run with matching settings, the subscriber repeatedly logs a line containing `listener` and `I heard:` followed by the publisher's configured text. If you left the initial settings unchanged, that text is `Hello from ROS 2!`.

Keep both nodes running. In a third terminal, source the same ROS 2 distribution and workspace overlay for graph inspection:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
cd "$COLCON_WS"
source install/setup.bash
```

## Diagnose missing messages

If the subscriber does not log a message, check one condition at a time in this exact order from the third terminal:

1. After changing either starter file, stop the affected node, rebuild `ros2_pubsub`, source the overlay in its terminal, and restart it.

2. Use `ros2 node list` to confirm that both `/talker` and `/listener` are still running.

3. Use `ros2 node info /talker` and `ros2 node info /listener`. The relevant publisher and subscription entries must both contain `/chatter: std_msgs/msg/String`. If the topic names differ, check `TOPIC_NAME` and `TOPIC_TO_SUBSCRIBE`; if the message types differ, set both `MSG_TYPE` values to `String`.

4. Run `ros2 topic echo --once /chatter`. If it prints a message, the publisher is working and the remaining problem is in the subscriber configuration.

5. Confirm that both node terminals use the same `ROS_DOMAIN_ID` and that neither command remapped one side to a different topic.

The basic publisher and subscriber use compatible default QoS settings. Later sensor nodes may require matching QoS policies as well.

## Further reading

The ROS 2 Jazzy tutorial on [writing a simple Python publisher and subscriber](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Writing-A-Simple-Py-Publisher-And-Subscriber.html) includes another subscriber example.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
